# Divisao -- treino / validacao / teste

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


def encontrar_raiz_repo(marcadores=(".git", "src")):
    caminho = Path.cwd()
    for pasta in [caminho, *caminho.parents]:
        if any((pasta / m).exists() for m in marcadores):
            return pasta
    raise FileNotFoundError(f"Nao achei nenhum de {marcadores} subindo a partir de {caminho}")


RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.max_colwidth", 120)


## Carregando o resultado da estruturacao

In [4]:
df = pd.read_csv(RAIZ / "data/processed/features_estruturadas.csv", index_col=0)
print(f"{len(df)} genomas, {df.shape[1]} colunas")
df.head()


2148 genomas, 9 colunas


,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,gc_diff_plasmid_cromossomo
sample,,,,,,,,,
GCA_000316425.1,15,0.000000,0.0,0.0,0.066667,0.933333,0.133333,0,-2.92
GCA_000355215.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,-2.10
GCA_000355195.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,-1.50
GCA_000355235.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.058824,2,-1.13
GCA_000355255.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.176471,0,-1.17


## Split 70/20/10

In [9]:
RANDOM_STATE = 42

df_train, df_temp = train_test_split(df, test_size=0.3, random_state=RANDOM_STATE)
df_valid, df_test = train_test_split(df_temp, test_size=0.35, random_state=RANDOM_STATE)

print(f"treino:     {len(df_train)} ({100*len(df_train)/len(df):.0f}%)")
print(f"validacao:  {len(df_valid)} ({100*len(df_valid)/len(df):.0f}%)")
print(f"teste:      {len(df_test)} ({100*len(df_test)/len(df):.0f}%)")


treino:     1503 (70%)
validacao:  419 (20%)
teste:      226 (11%)


## Conferindo que os 3 conjuntos nao ficaram muito diferentes

In [11]:
FEATURES = [
    "n_beta_lactam_total", "pct_esbl", "pct_A_serino_carbapenemase", "pct_BD_carbapenemase",
    "pct_C_ampc", "pct_nao_hidrolise", "pct_beta_lactam_plasmidial",
    "n_aminoglicosideo",
]

comparacao_splits = pd.concat({
    "treino": df_train[FEATURES].mean(),
    "validacao": df_valid[FEATURES].mean(),
    "teste": df_test[FEATURES].mean(),
}, axis=1).round(3)
comparacao_splits


,treino,validacao,teste
n_beta_lactam_total,14.667,14.790,14.345
pct_esbl,0.120,0.110,0.117
pct_A_serino_carbapenemase,0.032,0.028,0.030
pct_BD_carbapenemase,0.054,0.061,0.061
pct_C_ampc,0.046,0.052,0.050
pct_nao_hidrolise,0.726,0.727,0.718
pct_beta_lactam_plasmidial,0.241,0.214,0.235
n_aminoglicosideo,3.455,3.263,3.341


## Salvando os 3 conjuntos

In [12]:
pasta_saida = RAIZ / "data/processed/df_estruturada"
pasta_saida.mkdir(parents=True, exist_ok=True)

df_train.to_csv(pasta_saida / "df_train.csv")
df_valid.to_csv(pasta_saida / "df_valid.csv")
df_test.to_csv(pasta_saida / "df_test.csv")
print(f"Salvos em {pasta_saida}: df_train.csv, df_valid.csv, df_test.csv")


Salvos em /usr/src/myapp/data/processed/df_estruturada: df_train.csv, df_valid.csv, df_test.csv
